# 02 - Baseline Model

SE4050 Deep Learning Assignment - Brain Tumor Classification

This notebook trains and evaluates a classical machine learning baseline (e.g., SVM / Random Forest).

In [ ]:
import sys
sys.path.append('..')

import torch
from torch import nn, optim

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from src.dataset import split_dataset, create_dataloaders
from src.models import SimpleCNN
from src.utils import (
    set_seed,
    get_device,
    extract_hog_features,
    evaluate_predictions,
    train_model,
    get_predictions,
    save_checkpoint,
)

set_seed(42)
device = get_device()
print(f"Using device: {device}")

DATA_DIR = "../data"

## Load & Split the Dataset

Split into Train (70%) / Validation (15%) / Test (15%) with a fixed seed (42). The test set is
kept completely unseen and is not used anywhere in this baseline notebook.

In [ ]:
train_ds, val_ds, test_ds, classes = split_dataset(
    DATA_DIR, val_split=0.15, test_split=0.15, seed=42
)

print(f"Classes: {classes}")
print(f"Train size: {len(train_ds)} | Val size: {len(val_ds)} | Test size: {len(test_ds)}")

## Traditional Machine Learning Baseline

Extract HOG (Histogram of Oriented Gradients) features from the train/val splits, then train
an SVM and a Random Forest classifier for comparison against the CNN.

In [ ]:
X_train, y_train = extract_hog_features(train_ds)
X_val, y_val = extract_hog_features(val_ds)

print(f"Train features: {X_train.shape} | Val features: {X_val.shape}")

In [ ]:
svm_model = SVC(kernel="rbf", C=10, gamma="scale", random_state=42)
svm_model.fit(X_train, y_train)

svm_val_preds = svm_model.predict(X_val)
svm_metrics = evaluate_predictions(y_val, svm_val_preds, classes, title="SVM Baseline")

In [ ]:
rf_model = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_val_preds = rf_model.predict(X_val)
rf_metrics = evaluate_predictions(y_val, rf_val_preds, classes, title="Random Forest Baseline")

## Deep Learning Baseline: SimpleCNN

Train the `SimpleCNN` architecture (from `src/models.py`) on the augmented train split, using
DataLoaders built from the same reproducible 70/15/15 split.

In [ ]:
train_loader, val_loader, test_loader, classes = create_dataloaders(
    DATA_DIR, batch_size=32, val_split=0.15, test_split=0.15, seed=42
)

cnn_model = SimpleCNN(num_classes=len(classes)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=1e-3)

history = train_model(cnn_model, train_loader, val_loader, criterion, optimizer, device, epochs=10)

In [ ]:
y_true_cnn, y_pred_cnn = get_predictions(cnn_model, val_loader, device)
cnn_metrics = evaluate_predictions(y_true_cnn, y_pred_cnn, classes, title="SimpleCNN")

## Save the SimpleCNN Checkpoint

Persist the trained weights to `outputs/` for later reuse in the deep learning notebook.

In [ ]:
save_checkpoint(cnn_model, "../outputs/simple_cnn_baseline.pth")
print("Checkpoint saved to outputs/simple_cnn_baseline.pth")